In [ ]:
import numpy as np
import seaborn as sns
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
with open('./processed_data/heatmap_em.pkl', 'rb') as f:
    result_em = pickle.load(f)

In [ ]:
result_em['mat'].shape

In [ ]:
plt.figure(figsize=(12,10))
sns.heatmap(result_em['mat'], center=0.205, vmin=0, vmax=0.41, fmt='.3f', cmap=sns.color_palette("vlag_r", as_cmap=True), annot=True).invert_yaxis()
plt.xlabel(r'$\beta_{em}$')
plt.ylabel(r'$\sigma_{em}$')

In [ ]:
df_em = []
for i in range(10):
    for j in range(10):
        df_em.append((i,j, result_em['mat'][i,j]))
df_em = pd.DataFrame(df_em, columns=['sigma', 'beta', 'values'])

In [ ]:
# Prepare the data.
X = df_em.loc[df_em['beta']<9][['sigma', 'beta']]
y = df_em.loc[df_em['beta']<9][['values']]

# Create and fit the linear regression model.
model = LinearRegression()
model.fit(X, y)

# Print the fitted coefficients.
print("Slope:", model.coef_)
print("Intercept:", model.intercept_)
r_squared = r2_score(y_true=y, y_pred=model.predict(X))
print("R-squared:", r_squared)


# Prepare the data.
X = df_em[['sigma', 'beta']]
y = df_em[['values']]

# Create and fit the linear regression model.
model = LinearRegression()
model.fit(X, y)

# Print the fitted coefficients.
print("Slope:", model.coef_)
print("Intercept:", model.intercept_)
r_squared = r2_score(y_true=y, y_pred=model.predict(X))
print("R-squared:", r_squared)

In [ ]:
sns.lineplot(df_em, x='sigma', y='values', hue='beta', hue_order=[0,1,2,3,4,5,6,7,8,9])

In [ ]:
sns.lineplot(df_em, x='beta', y='values', hue='sigma', hue_order=[0,1,2,3,4,5,6,7,8,9])

In [ ]:
with open('./processed_data/heatmap_sim.pkl', 'rb') as f:
    result_sm = pickle.load(f)

In [ ]:
result_sm['mat'].shape

In [ ]:
plt.figure(figsize=(12,10))
sns.heatmap(result_sm['mat'].T, center=0.205, vmin=0, vmax=0.41, fmt='.3f', cmap=sns.color_palette("vlag_r", as_cmap=True), annot=True).invert_yaxis()
plt.xticks(np.array([0,2,4,6,8,10])+0.5, [5, 6, 7 ,8, 9, 10])
plt.yticks(np.array([0,2,4,6,8,10])+0.5, np.linspace(0.018, 0.02, 11)[::2])
plt.xlabel(r'$\beta_{th}$')
plt.ylabel(r'$\sigma_{th}$')


In [ ]:
df_th = []
for i in range(10):
    for j in range(10):
        df_th.append((i,j, result_sm['mat'].T[i,j]))
df_th = pd.DataFrame(df_th, columns=['sigma', 'beta', 'values'])

In [ ]:
# Prepare the data.
X = df_th.loc[df_th['beta']<7][['sigma', 'beta']]
y = df_th.loc[df_th['beta']<7][['values']]

# Create and fit the linear regression model.
model = LinearRegression()
model.fit(X, y)

# Print the fitted coefficients.
print("Slope:", model.coef_)
print("Intercept:", model.intercept_)
r_squared = r2_score(y_true=y, y_pred=model.predict(X))
print("R-squared:", r_squared)


# Prepare the data.
X = df_th[['sigma', 'beta']]
y = df_th[['values']]

# Create and fit the linear regression model.
model = LinearRegression()
model.fit(X, y)

# Print the fitted coefficients.
print("Slope:", model.coef_)
print("Intercept:", model.intercept_)
r_squared = r2_score(y_true=y, y_pred=model.predict(X))
print("R-squared:", r_squared)

In [ ]:
parameters_em = np.array([-0.01588441, 0.01914977])
parameters_th = np.array([-0.00898874, 0.05315714])


In [ ]:
# sigma ,beta -- non-extreme cases
def get_mapping(parameters_em, parameters_th):
    sigma_mapping = parameters_em[0]/parameters_th[0]*0.0002
    beta_mapping = parameters_em[1]/parameters_th[1]*0.5
    return sigma_mapping,beta_mapping
def mapping_function(para, sigma_em, beta_em):
    return 0.018+para[0]*sigma_em, 5+para[1]*beta_em

In [ ]:
para = get_mapping(parameters_em, parameters_th)
mapping_function(para,1,1)

In [ ]:
# Extreme cases: the rightmost empirical cells.
c_em = np.max(df_em[df_em['beta']==9]['values'].to_numpy())-np.min(df_em[df_em['beta']==9]['values'].to_numpy())
c_th = np.max(df_th[df_th['beta']==8]['values'].to_numpy())-np.min(df_th[df_th['beta']==8]['values'].to_numpy())

In [ ]:
def mapping_function(para, c_th, c_em, sigma_em, beta_em):
    if beta_em < 9:
        return 0.018+para[0]*sigma_em, 5+para[1]*beta_em
    else: 
        return 0.018+c_em/c_th*0.0002*sigma_em, 9

In [ ]:
def mapping_function_idx(para, c_th, c_em, sigma_em, beta_em):
    if beta_em < 9:
        return para[0]*sigma_em/0.0002, para[1]*beta_em/0.5
    else: 
        return c_em/c_th*sigma_em, 8

In [ ]:
mapping_function(para=para, c_th=c_th, c_em=c_em, sigma_em=9, beta_em=8)

In [ ]:
mapping_function_idx(para=para, c_th=c_th, c_em=c_em, sigma_em=1, beta_em=1)

In [ ]:
age_range = ['18-23','24-30','31-40','41-50','51-60','60+']
for aa in age_range:
    with open('./processed_data/age_range_{}_em.pkl'.format(aa), 'rb') as f:
        dist = pickle.load(f)
    plt.figure(figsize=(12,10))
    sns.heatmap(dist/np.sum(dist), annot=True).invert_yaxis()
    plt.xlabel(r'$\beta_{em}$')
    plt.ylabel(r'$\sigma_{em}$')
    plt.title(aa)

In [ ]:
def get_samples(overall_num):
    from collections import Counter
    age_range = ['18-23','24-30','31-40','41-50','51-60','60+']
    with open('./processed_data/age_range_all_dist.pkl','rb') as f:
        results = pickle.load(f)
    age_dist = age_dist['age_dist']
    overall_population = np.random.choice(np.arange(6), p=age_dist,size=overall_num)
    overall_population = Counter(overall_population)
    all_samples = {}
    for i in range(6):
        age_rr = age_range[i]
        cnt = overall_population[i]
        dist = results[age_range[i]]
        age_samples = np.random.choice(a=np.arange(100), p=dist.flatten(), size=cnt)
        sigma = age_samples//10 
        beta = age_samples%10
        age_samples = [z for z in zip(sigma, beta)]
        all_samples[age_rr] = {'cnt':cnt, 'samples': age_samples} 
    return all_samples




In [ ]:
samples = get_samples(100000)

In [ ]:
with open('./processed_data/samples_100000.pkl','wb') as f:
    pickle.dump(samples, f)

In [ ]:
age_range = ['18-23','24-30','31-40','41-50','51-60','60+']
for aa in age_range:
    dist = np.zeros((10,10))
    ss = samples[aa]['samples']
    for s in ss:
        dist[s[0],s[1]] += 1
    plt.figure(figsize=(12,10))
    sns.heatmap(dist, annot=True).invert_yaxis()
    plt.xlabel(r'$\beta_{th}$')
    plt.ylabel(r'$\sigma_{th}$')
    plt.title(aa)

In [ ]:
age_range = ['18-23','24-30','31-40','41-50','51-60','60+']
results_samples = {}
for aa in age_range:
    ss = samples[aa]['samples']
    rs = []
    for s in ss:
        sigma_th, beta_th = mapping_function_idx(para=para, c_th=c_th, c_em=c_em, sigma_em=s[0], beta_em=s[1])
        sigma_th = np.clip(np.round(sigma_th), 0, 10).astype('int')
        beta_th = np.clip(np.round(beta_th), 0, 10).astype('int')
        rs.append((sigma_th,beta_th))
    results_samples[aa] = rs

In [ ]:
results_samples

In [ ]:
with open('./processed_data/samples_results_100000.pkl','wb') as f:
    pickle.dump(results_samples, f)

In [ ]:
# Plot the distribution of inequality across groups.
plt.figure(figsize=(8,6))
c_adopted = criteria_list[3]
print(c_adopted, population_list[3])
data['indicator'] = data['m_ratio'].transform(lambda x: 1 if x<c_adopted else 0)
sns.barplot(data, x='age_range', y='indicator', order=['18-23','24-30','31-40','41-50','51-60','60+'], errorbar=bernoulli_ci, palette=sns.color_palette("Blues", 6), capsize=0.2)
plt.hlines(y=population_list[3], xmin=-0.75, xmax=5.75, lw=5, color='gray', ls='--', alpha=0.5, label='Avg')
plt.ylabel('Propensity for Deep ICs')
plt.xlabel('Age')
plt.xlim(-0.75, 5.75)
plt.legend()
plt.savefig('./figures/figure1/IC_disparity.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)